# Education

Subnational **Education Index** from the **[Global Data Lab](https://globaldatalab.org/shdi/) Subnational HDI database (SHDI v8.3)** — a `[0, 1]` composite of mean and expected years of schooling, published for ~1 800 first- and second-level administrative regions across ~180 countries. This is the finest openly-available *global* education layer: truly sub-national wherever GDL provides an admin-1 split (US states, Indian states, Brazilian states, most of Europe), country-flat where they only publish the national total (many small states).

GDL's own download portal requires a free account, so we fetch the same v8.3 CSV and the matching GDL Shapefiles v6.5 from the **[Zenodo redistribution](https://zenodo.org/records/17467221)** (Smits & Permanyer, released September 2025). Values are joined onto GDL polygons by `GDLCODE`, then rasterised onto the shared 0.5° atlas grid via `dominant_region_mask` — the same fractional-coverage pattern `03_region_mask.ipynb` uses for Natural Earth countries, just with many more, smaller polygons.

The Education Index is already `[0, 1]` with higher = more education, so **no sign inversion** is needed. Ocean is masked via `is_land` from `grid.nc`. Regions absent from GDL stay `NaN`; `weighted_score` renormalises per-cell weights so the missing layer neither penalises nor discards a cell. Downgraded to 🟡 Tier B in `DATASETS.md` — subnational where possible, but still polygon-flat within each region and not truly gridded.

> Version note: GDL currently ships v10.2 behind login, but only v8.3 is on the public Zenodo mirror. The two are structurally identical (same `GDLCODE`, same `Educ` column) — swap the URL below once a newer version is mirrored.

In [7]:
import zipfile

import geopandas as gpd
import numpy as np
import pandas as pd
import regionmask
import xarray as xr

from common import RAW_DIR, dominant_region_mask, download, load_grid, plot_map, save_variable

VARIABLE = 'education'
variable_raw = RAW_DIR / VARIABLE
variable_raw.mkdir(parents=True, exist_ok=True)

## 1. Fetch raw data

Two artefacts from the Zenodo mirror of GDL v8.3:

- `Subnational HDI Data v8.3.csv` (~12 MB) — one row per `(GDLCODE, year)` with `Educ` as the education-index column.
- `GDL Shapefiles V6.5.zip` (~340 MB) — the subnational polygon geometry the CSV keys onto via `gdlcode`.

Both are cached under `variable_raw`; the download only runs on first execution. Zenodo occasionally rate-limits fresh IPs — if the fetch fails with an HTTP 403, wait a few minutes and re-run this cell.

In [8]:
SHDI_URL = 'https://zenodo.org/records/17467221/files/Subnational%20HDI%20Data%20v8.3.csv?download=1'
SHAPES_URL = 'https://zenodo.org/records/17467221/files/GDL%20Shapefiles%20V6.5.zip?download=1'

shdi_csv = variable_raw / 'Subnational_HDI_Data_v8.3.csv'
shapes_zip = variable_raw / 'GDL_Shapefiles_V6.5.zip'

if not shdi_csv.exists():
    await download(SHDI_URL, shdi_csv)
if not shapes_zip.exists():
    await download(SHAPES_URL, shapes_zip)

print(f'{shdi_csv.name}: {shdi_csv.stat().st_size / 1024**2:.1f} MB')
print(f'{shapes_zip.name}: {shapes_zip.stat().st_size / 1024**2:.0f} MB')

Subnational_HDI_Data_v8.3.csv: 11.2 MB
GDL_Shapefiles_V6.5.zip: 322 MB


## 2. Parse GDL Education Index

The SHDI CSV has one row per `(GDLCODE, year)` with sub-index columns (`shdi`, `healthindex`, `edindex`, `incindex`, ...). Keep only rows with a non-null `edindex` (the Educational Index — a `[0, 1]` composite of expected + mean years of schooling), take the latest available year per region, and index by `GDLCODE` for the join in the next cell.

The CSV mixes national (`level == 'National'`, `GDLCODE` ending in `t`) and subnational rows — we keep both here and let the shapefile decide which geometry each `GDLCODE` maps to.

In [ ]:
shdi = pd.read_csv(shdi_csv)
print(f'{len(shdi):,} rows, columns: {list(shdi.columns)}')


shdi = shdi.dropna(subset=['edindex'])
latest = (
    shdi.sort_values('year')
        .groupby('gdlcode', as_index=False)
        .tail(1)
        .set_index('gdlcode')
)

print(f"{len(latest):,} regions with edindex (years {int(latest['year'].min())}–{int(latest['year'].max())}, "
      f"range [{latest['edindex'].min():.3f}, {latest['edindex'].max():.3f}])")
latest[['country', 'region', 'level', 'year', 'edindex']].head()

61,179 rows, columns: ['isocode3', 'country', 'continent', 'datasource', 'year', 'gdlcode', 'level', 'region', 'sgdi', 'shdif', 'shdim', 'healthindexf', 'healthindexm', 'edindexf', 'edindexm', 'incindexf', 'incindexm', 'shdi', 'healthindex', 'edindex', 'incindex', 'lifexp', 'lifexpf', 'lifexpm', 'esch', 'eschf', 'eschm', 'msch', 'mschf', 'mschm', 'lgnic', 'lgnicf', 'lgnicm', 'pop']
      isocode3      country     continent  datasource  year  gdlcode  \
0          AFG  Afghanistan  Asia/Pacific         NaN  1990     AFGt   
1          AFG  Afghanistan  Asia/Pacific         NaN  1991     AFGt   
2          AFG  Afghanistan  Asia/Pacific         NaN  1992     AFGt   
3          AFG  Afghanistan  Asia/Pacific         NaN  1993     AFGt   
4          AFG  Afghanistan  Asia/Pacific         NaN  1994     AFGt   
...        ...          ...           ...         ...   ...      ...   
61174      ZWE     Zimbabwe        Africa         NaN  2018  ZWEr110   
61175      ZWE     Zimbabwe        Afri

,country,region,level,year,edindex
gdlcode,,,,,
IDNr119,Indonesia,East Timor,Subnat,2001,0.392
BGDr206,Bangladesh,"Feni, Lakshmipur, Noakhali",Subnat,2022,0.573
BENr104,Benin,Mono (incl Couffo),Subnat,2022,0.403
ARMr106,Armenia,Kotayk,Subnat,2022,0.781
ARMr109,Armenia,Vayots Dzor,Subnat,2022,0.768


## 3. Rasterise onto the 0.5° grid

Load the GDL subnational shapefile straight from the zip via `geopandas`, normalise the join column name (the shapefile ships as `gdlcode` lowercase), and drop polygons for which we have no Education Index. Then `dominant_region_mask` picks the region with the largest fractional coverage per 0.5° cell (same helper `03_region_mask.ipynb` uses for countries — internally it oversamples each atlas cell into 10×10 sub-pixels with `rasterio.features.rasterize` and takes the mode) and we look up each region's `edindex` value.

With ~1 800 polygons the mask itself completes in a fraction of a second; the bulk of the runtime on first execution is the initial shapefile parse.

In [ ]:
grid = load_grid()

shapes = gpd.read_file(f'zip://{shapes_zip}').to_crs('EPSG:4326')
# Field name varies by shapefile version (gdlcode / GDLcode / GDLCODE); pick whichever matches.
code_col = next(c for c in shapes.columns if c.lower() == 'gdlcode')
shapes = shapes.rename(columns={code_col: 'GDLCODE'})
shapes['edindex'] = shapes['GDLCODE'].map(latest['edindex'])
shapes = shapes.dropna(subset=['edindex']).reset_index(drop=True)
print(f'{len(shapes):,} polygons with an Education Index')

regions = regionmask.from_geopandas(shapes, overlap=False)
region_num = dominant_region_mask(regions, grid.lon, grid.lat)

# region_num carries the region *numbers* (== row index after reset_index) as float with NaN
# for cells outside every polygon; look up edindex per cell and mask ocean.
edindex_by_num = shapes['edindex'].to_numpy(dtype='float32')
idx = region_num.values
valid = np.isfinite(idx)
raster = np.full(idx.shape, np.nan, dtype='float32')
raster[valid] = edindex_by_num[idx[valid].astype(np.int64)]

values = xr.DataArray(
    raster,
    coords={'lat': grid.lat, 'lon': grid.lon},
    dims=('lat', 'lon'),
    name=VARIABLE,
).where(grid.is_land == 1)
values.attrs['source'] = 'GDL Subnational HDI v8.3 (edindex), GDL Shapefiles v6.5'
values.attrs['units'] = 'index [0, 1], higher = better'

covered = int(values.notnull().sum())
land = int((grid.is_land == 1).sum())
print(f'{covered:,} land cells with a value ({100 * covered / land:.1f}% of land)')
values

## 4. Plot

In [ ]:
plot_map(values, cmap='RdYlGn', robust=True)

## 5. Save

In [ ]:
out = save_variable(values, VARIABLE)
print(f'wrote {out}')